In [4]:
# =====================================================================
#  NETWORK INTRUSION DETECTION SYSTEM (IDS) SIMULATION  -  single Colab cell
#  Defensive education project: ALL traffic is synthetic flow *records*.
#  Nothing here sends packets, scans, or touches any real network.
#  Run: paste into ONE Colab cell -> Shift+Enter.
#  Pipeline: simulator -> feature extraction -> rules + anomaly + ML
#            -> hybrid risk -> alerts -> SQLite -> correlation -> dashboard
# =====================================================================
import os, json, time, uuid, sqlite3, ipaddress, warnings
from collections import deque, defaultdict
from datetime import datetime, timedelta
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
try:
    from IPython.display import display, HTML, clear_output
except ImportError:                       # lets the script run outside notebooks
    display, HTML, clear_output = print, (lambda s: s), (lambda **k: None)
warnings.filterwarnings("ignore")

# ------------------------------ CONFIG -------------------------------
CONFIG = dict(
    seed=42, n_flows=6000, suspicious_share=0.20,           # dataset
    live_flows=700, live_mode="mixed", live_speed="instant", # mode: normal|mixed ; speed: slow|fast|instant
    use_ml=True,
    weights_ml={"rule": .4, "anomaly": .3, "ml": .3},        # configurable hybrid weights
    weights_no_ml={"rule": .6, "anomaly": .4},
    alert_min_risk=21, corr_window=60, window_seconds=60,
    thresholds=dict(conn_rate=10.0, failed_count=10, failure_ratio=0.6, unique_ports=10,
                    syn_count=100, syn_ratio=0.5, bytes=20_000_000),
)
PROTOCOLS = {"TCP", "UDP", "ICMP"}
STATUSES = ["NEW", "INVESTIGATING", "RESOLVED", "FALSE_POSITIVE"]
TRANSITIONS = {"NEW": {"INVESTIGATING", "FALSE_POSITIVE"}, "INVESTIGATING": {"RESOLVED", "FALSE_POSITIVE"},
               "RESOLVED": set(), "FALSE_POSITIVE": set()}
SEV_ORDER = ["INFO", "LOW", "MEDIUM", "HIGH", "CRITICAL"]
SEV_SCORE = {"INFO": 20, "LOW": 40, "MEDIUM": 60, "HIGH": 80, "CRITICAL": 95}
KNOWN_PORTS = {20, 21, 22, 25, 53, 80, 110, 123, 143, 443, 465, 587, 993, 995, 3306, 5432, 8080, 8443}
PORT_NAMES = {443: "443/HTTPS", 80: "80/HTTP", 53: "53/DNS", 22: "22/SSH", 25: "25/SMTP", 587: "587/SMTP", 993: "993/IMAPS",
              143: "143/IMAP", 3306: "3306/MySQL", 5432: "5432/Postgres", 8080: "8080/HTTP-alt", 123: "123/NTP", 21: "21/FTP"}

# ------------------- 1. SYNTHETIC TRAFFIC (DATA ONLY) ----------------
CLIENTS = [f"192.0.2.{i}" for i in range(10, 60)]         # RFC 5737 documentation ranges
SERVERS = [f"198.51.100.{i}" for i in range(10, 40)]
SUSP_SOURCES = ["203.0.113.45", "203.0.113.77", "203.0.113.12", "203.0.113.201", "203.0.113.150", "192.0.2.88", "192.0.2.131"]
NORMAL_SC = ["NORMAL_WEB", "NORMAL_DNS", "NORMAL_SSH", "NORMAL_EMAIL", "NORMAL_DATABASE"]
MP = "MULTI_PORT_PROBING_PATTERN"
SUSP_SC = ["HIGH_CONNECTION_RATE", "REPEATED_FAILED_CONNECTIONS", MP, "SYN_HEAVY_PATTERN", "UNUSUAL_PORT_ACTIVITY", "HIGH_TRAFFIC_VOLUME"]

def make_flow(rng, scn, ts, src=None, dst=None, dport=None):
    """Build ONE synthetic flow record (a dict of numbers) for a scenario. No packets are created."""
    I = lambda a, b: int(rng.integers(a, b + 1)); U = rng.uniform
    proto, failed, rst, syn = "TCP", 0, 0, 0
    if scn.startswith("NORMAL"):
        failed = 1 if rng.random() < 0.02 else 0; rst = failed
    if scn == "NORMAL_WEB":
        dport = dport or int(rng.choice([443, 443, 443, 80])); pk = I(10, 80); size = U(400, 1200); dur = U(.5, 8); conn = I(1, 4); syn = I(1, 3)
    elif scn == "NORMAL_DNS":
        proto, dport = "UDP", 53; pk = I(2, 6); size = U(60, 120); dur = U(.02, .5); conn = I(1, 3)
    elif scn == "NORMAL_SSH":
        dport = 22; pk = I(30, 300); size = U(80, 300); dur = U(10, 600); conn = I(1, 2); syn = I(1, 2)
    elif scn == "NORMAL_EMAIL":
        dport = int(rng.choice([587, 993, 25, 143])); pk = I(10, 100); size = U(300, 1400); dur = U(1, 20); conn = I(1, 3); syn = I(1, 2)
    elif scn == "NORMAL_DATABASE":
        dport = int(rng.choice([5432, 3306])); pk = I(20, 200); size = U(100, 900); dur = U(1, 60); conn = I(1, 5); syn = I(1, 3)
    elif scn == "HIGH_CONNECTION_RATE":
        conn = I(80, 250); dur = U(1, 5); pk = conn * I(2, 4); size = U(60, 200); syn = int(conn * U(.8, 1)); failed = int(conn * U(0, .15))
        dport = dport or int(rng.choice([443, 80, 8080]))
    elif scn == "REPEATED_FAILED_CONNECTIONS":
        conn = I(20, 60); dur = U(3, 30); failed = int(conn * U(.7, .95)); syn = conn; rst = failed; pk = conn * I(2, 3); size = U(60, 100)
        dport = dport or int(rng.choice([22, 3306, 5432, 443]))
    elif scn == MP:
        dport = dport or I(1, 1024); pk = I(2, 6); size = U(40, 80); dur = U(.02, .4); conn = I(1, 3); syn = conn; failed = I(1, conn); rst = failed
    elif scn == "SYN_HEAVY_PATTERN":
        syn = I(150, 600); pk = syn + I(0, 30); size = U(40, 70); dur = U(1, 5); conn = I(3, 8); failed = I(0, 2); rst = I(0, 20)
        dport = dport or int(rng.choice([443, 80, 22]))
    elif scn == "UNUSUAL_PORT_ACTIVITY":
        dport = int(rng.choice([4444, 31337, 6667, 1337, 8888, 9001, 12345])); pk = I(20, 200); size = U(100, 800); dur = U(2, 60); conn = I(1, 5); syn = I(1, 3)
        failed = int(rng.random() < .2)
    elif scn == "HIGH_TRAFFIC_VOLUME":
        dport = int(rng.choice([443, 21, 80])); pk = I(40000, 400000); size = U(800, 1400); dur = U(20, 120); conn = I(1, 5); syn = I(1, 3)
    else:
        raise ValueError(scn)
    normal = scn.startswith("NORMAL")
    by = int(pk * size)
    return dict(timestamp=ts.strftime("%Y-%m-%d %H:%M:%S"),
                source_ip=src or str(rng.choice(CLIENTS if normal else SUSP_SOURCES)),
                destination_ip=dst or str(rng.choice(SERVERS)), source_port=I(49152, 65535), destination_port=int(dport),
                protocol=proto, packet_count=int(pk), byte_count=by, duration_seconds=round(float(dur), 3),
                connection_count=int(conn), failed_connection_count=int(min(failed, conn)), syn_count=int(syn), rst_count=int(rst),
                average_packet_size=round(by / max(pk, 1), 1), label="NORMAL" if normal else "SUSPICIOUS", scenario_type=scn)

def make_burst(rng, t0, src, dst, k):
    """One source touching k different destination ports = multi-port probing-like PATTERN (records only)."""
    out, t = [], t0
    for p in rng.choice(np.arange(1, 1025), size=k, replace=False):
        t += timedelta(seconds=float(rng.uniform(.3, 1.5))); out.append(make_flow(rng, MP, t, src, dst, int(p)))
    return out

def generate_dataset(n=6000, share=.2, seed=42, span_h=6):
    rng = np.random.default_rng(seed)
    start = datetime.now().replace(microsecond=0) - timedelta(hours=1 + span_h)
    rt = lambda: start + timedelta(seconds=float(rng.uniform(0, span_h * 3600)))
    n_s = int(n * share); n_mp = n_s // 6; rows = []
    for _ in range(n - n_s): rows.append(make_flow(rng, str(rng.choice(NORMAL_SC, p=[.5, .2, .1, .1, .1])), rt()))
    single = [s for s in SUSP_SC if s != MP]
    for i in range(n_s - n_mp): rows.append(make_flow(rng, single[i % 5], rt()))
    made = 0
    while made < n_mp:
        k = min(int(rng.integers(18, 32)), n_mp - made); made += k
        rows += make_burst(rng, rt(), str(rng.choice(SUSP_SOURCES)), str(rng.choice(SERVERS)), k)
    df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    df.insert(0, "flow_id", [f"F-{i + 1:06d}" for i in range(len(df))])
    return df

# ------------------- 2. TRAFFIC SIMULATOR (generator) ----------------
def traffic_simulator(mode="mixed", speed="fast", n=700, seed=7):
    """Continuously yields synthetic flow dicts. mode: normal|mixed. speed: slow|fast|instant. NEVER sends packets."""
    rng = np.random.default_rng(seed); t = datetime.now().replace(microsecond=0) - timedelta(minutes=30)
    delay = {"slow": .5, "fast": .02, "instant": 0}[speed]; i = 0
    while i < n:
        batch = []
        if mode == "mixed" and rng.random() < .05:
            scn = str(rng.choice(SUSP_SC))
            if scn == MP: batch = make_burst(rng, t, str(rng.choice(SUSP_SOURCES)), str(rng.choice(SERVERS)), int(rng.integers(14, 27)))
            else: batch = [make_flow(rng, scn, t)]
        else:
            batch = [make_flow(rng, str(rng.choice(NORMAL_SC, p=[.5, .2, .1, .1, .1])), t)]
        for fl in batch:
            i += 1; fl["flow_id"] = f"L-{i:06d}"; yield fl
            if delay: time.sleep(delay)
        t += timedelta(seconds=float(rng.exponential(2.5)) + 0.5)

# ------------------- 3. FEATURE ENGINEERING --------------------------
class WindowTracker:
    """Remembers recent (dst_ip, dst_port) per source in a sliding window -> port/IP diversity features."""
    def __init__(self, window=60): self.w, self.h = window, defaultdict(deque)
    def update(self, src, ts, dip, dport):
        q = self.h[src]; q.append((ts, dip, dport))
        while q and ts - q[0][0] > self.w: q.popleft()
        return len({p for _, _, p in q}), len({i for _, i, _ in q})

def _ip_ok(x):
    try: ipaddress.ip_address(str(x).strip()); return True
    except ValueError: return False
def _port_ok(p, lo=0):
    try:
        if isinstance(p, float) and not p.is_integer(): return False
        return lo <= int(p) <= 65535
    except (TypeError, ValueError): return False

def extract_network_features(flow, tracker=None):
    """Validate a raw flow and derive the 15 IDS features. Never raises; bad input -> valid=False + reasons."""
    errors, imputed = [], []
    if not _ip_ok(flow.get("source_ip")): errors.append("invalid source_ip")
    if not _ip_ok(flow.get("destination_ip")): errors.append("invalid destination_ip")
    proto = str(flow.get("protocol", "")).upper()
    if proto not in PROTOCOLS: errors.append("unsupported protocol")
    if not _port_ok(flow.get("source_port"), 0): errors.append("invalid source_port")
    if not _port_ok(flow.get("destination_port"), 0 if proto == "ICMP" else 1): errors.append("invalid destination_port")
    try: ts = pd.Timestamp(flow.get("timestamp")).timestamp()
    except Exception: ts = None; errors.append("invalid timestamp")
    def num(k):                                             # missing / NaN / negative -> 0 (recorded)
        try: v = float(flow.get(k))
        except (TypeError, ValueError): v = float("nan")
        if not np.isfinite(v) or v < 0: imputed.append(k); return 0.0
        return v
    pk, by, du, cc, fc, sy, rs = (num(k) for k in ["packet_count", "byte_count", "duration_seconds", "connection_count",
                                                   "failed_connection_count", "syn_count", "rst_count"])
    d = max(du, .1)                                          # division-by-zero guard
    f = dict(packet_count=pk, byte_count=by, duration_seconds=du, bytes_per_second=by / d, packets_per_second=pk / d,
             average_packet_size=by / pk if pk > 0 else 0.0, connection_count=cc, failed_connection_count=fc,
             failure_ratio=min(fc / cc, 1.0) if cc > 0 else 0.0, syn_count=sy, rst_count=rs,
             syn_ratio=min(sy / pk, 1.0) if pk > 0 else 0.0, unique_destination_ports=1, unique_destination_ips=1,
             connection_rate=cc / max(du, 1.0))
    if not errors and tracker is not None and ts is not None:
        f["unique_destination_ports"], f["unique_destination_ips"] = tracker.update(
            str(flow["source_ip"]), ts, str(flow["destination_ip"]), int(flow["destination_port"]))
    if not errors: f["_dport"], f["_proto"] = int(flow["destination_port"]), proto   # used by the unusual-port rule
    f.update(valid=not errors, errors=errors, imputed=imputed)
    return f

def featurize_df(df):
    if df is None or df.empty: return pd.DataFrame()
    tr, out = WindowTracker(CONFIG["window_seconds"]), []
    for r in df.sort_values("timestamp").to_dict("records"):
        f = extract_network_features(r, tr)
        if f["valid"]: f.update(flow_id=r.get("flow_id"), label=r.get("label"), scenario_type=r.get("scenario_type")); out.append(f)
    return pd.DataFrame(out)

# ------------------- 4. SIGNATURE / RULE ENGINE ----------------------
RULE_DEFS = [  # (id, name, severity, description, predicate(features, thresholds), threshold text)
    ("IDS-001", "High Connection Rate", "HIGH", "Connection rate exceeded configured baseline.",
     lambda f, t: f["connection_rate"] >= t["conn_rate"], "connection_rate >= {conn_rate}/s"),
    ("IDS-002", "Repeated Failed Connections", "HIGH", "Repeated failed connection attempts from one source.",
     lambda f, t: f["failed_connection_count"] >= t["failed_count"] and f["failure_ratio"] >= t["failure_ratio"],
     "failed >= {failed_count} and ratio >= {failure_ratio}"),
    ("IDS-003", "Many Destination Ports", "HIGH", "One source contacted an unusually large number of destination ports.",
     lambda f, t: f["unique_destination_ports"] >= t["unique_ports"], "unique dst ports (60s) >= {unique_ports}"),
    ("IDS-004", "SYN-Heavy Behaviour", "HIGH", "SYN-dominated flow with very high SYN volume.",
     lambda f, t: f["syn_count"] >= t["syn_count"] and f["syn_ratio"] >= t["syn_ratio"], "syn >= {syn_count} and syn_ratio >= {syn_ratio}"),
    ("IDS-005", "Unusual Service Port", "MEDIUM", "Traffic to a port outside the approved service list.",
     lambda f, t: f.get("_dport", 0) not in KNOWN_PORTS and f.get("_proto") != "ICMP", "dst port not in approved list"),
    ("IDS-006", "Abnormally High Traffic Volume", "MEDIUM", "Flow volume far above normal baseline.",
     lambda f, t: f["byte_count"] >= t["bytes"], "bytes >= {bytes}"),
]
RULE_ENABLED = {r[0]: True for r in RULE_DEFS}

def analyze_flow(f, thresholds=None):
    """Signature engine: returns the list of rules matched. A match = 'investigate', NOT 'proven attack'."""
    t = {**CONFIG["thresholds"], **(thresholds or {})}
    return [dict(rule_id=i, name=n, severity=s, description=d) for i, n, s, d, p, _ in RULE_DEFS if RULE_ENABLED[i] and p(f, t)]

def rule_risk(matches):
    if not matches: return 0.0
    return float(min(100, max(SEV_SCORE[m["severity"]] for m in matches) + 5 * (len(matches) - 1)))

# ------------------- 5. ANOMALY DETECTION (statistical) --------------
ANOM = {"packets_per_second": (True, .5), "bytes_per_second": (True, .5), "connection_rate": (True, .5),
        "failure_ratio": (False, .25), "unique_destination_ports": (False, 1.5)}   # (log-transform?, std floor)
BASE = {}
def _tr(k, v): return float(np.log1p(v)) if ANOM[k][0] else float(v)
def fit_baseline(normal_df):
    for k, (lg, floor) in ANOM.items():
        s = normal_df[k].map(lambda v: _tr(k, v)); BASE[k] = (float(s.mean()), max(float(s.std()), floor))
def anomaly_z(f): return {k: (_tr(k, f[k]) - BASE[k][0]) / BASE[k][1] for k in ANOM}
def calculate_anomaly_score(f):
    """0-100. Z-score vs. normal baseline (log-scaled rates), one-sided; z<=1.5 -> 0, z>=6 -> 100."""
    if not BASE: return 0.0
    s = [float(np.clip((z - 1.5) / 4.5, 0, 1)) * 100 for z in anomaly_z(f).values()]
    return round(0.65 * max(s) + 0.35 * float(np.mean(s)), 1)
def moving_average(vals, w=3): return pd.Series(vals, dtype=float).rolling(w, min_periods=1).mean().round(1).tolist()

# ------------------- 6. ML MODELS + HYBRID RISK ----------------------
ML_FEATURES = ["packet_count", "byte_count", "duration_seconds", "bytes_per_second", "packets_per_second",
               "connection_count", "failed_connection_count", "syn_count", "rst_count", "average_packet_size"]
LR = RF = ISO = None; MODEL_INFO = {}

def _m(y, p, name):
    return dict(model=name, accuracy=round(accuracy_score(y, p), 4), precision=round(precision_score(y, p, zero_division=0), 4),
                recall=round(recall_score(y, p, zero_division=0), 4), f1=round(f1_score(y, p, zero_division=0), 4))

def train_models(F):
    global LR, RF, ISO, MODEL_INFO
    X, y = F[ML_FEATURES], (F.label == "SUSPICIOUS").astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
    fit_baseline(F.loc[Xtr.index[ytr.values == 0]])                         # baseline from TRAIN-normal only (no leakage)
    LR = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(Xtr, ytr)
    RF = RandomForestClassifier(n_estimators=60, max_depth=12, class_weight="balanced", random_state=42, n_jobs=1).fit(Xtr, ytr)
    ISO = IsolationForest(n_estimators=100, contamination=.02, random_state=42).fit(Xtr[ytr == 0])   # unsupervised: normal only
    P = {"Logistic Regression": LR.predict(Xte), "Random Forest": RF.predict(Xte),
         "Isolation Forest (unsupervised)": (ISO.predict(Xte) == -1).astype(int)}
    models = [_m(yte, p, n) for n, p in P.items()]; cms = {n: confusion_matrix(yte, p, labels=[0, 1]).tolist() for n, p in P.items()}
    recs = F.loc[Xte.index].to_dict("records"); prob = RF.predict_proba(Xte)[:, 1] * 100
    mt = [analyze_flow(r) for r in recs]; an = [calculate_anomaly_score(r) for r in recs]
    hyb = [calculate_risk_score(rule_risk(m), a, p) for m, a, p in zip(mt, an, prob)]
    layers = {"Signature rules only": [int(bool(m)) for m in mt], "Anomaly only (score >= 50)": [int(a >= 50) for a in an],
              "ML only (Random Forest)": P["Random Forest"], "Hybrid (risk >= 41)": [int(h >= 41) for h in hyb]}
    MODEL_INFO = dict(models=models, compare=[_m(yte, np.array(p), n) for n, p in layers.items()], cm=cms["Random Forest"],
                      n_train=len(Xtr), n_test=len(Xte))

def ml_scores(f):
    X = pd.DataFrame([[f[k] for k in ML_FEATURES]], columns=ML_FEATURES)
    return dict(rf=float(RF.predict_proba(X)[0, 1] * 100), lr=float(LR.predict_proba(X)[0, 1] * 100),
                iso=int(ISO.predict(X)[0] == -1), iso_raw=float(-ISO.score_samples(X)[0]))

def calculate_risk_score(rule_s, anomaly_s, ml_p=None, weights=None):
    """Weighted hybrid score 0-100. ML off -> 60/40 (rule/anomaly); ML on -> 40/30/30. Weights live in CONFIG."""
    if ml_p is None: w = weights or CONFIG["weights_no_ml"]; r = w["rule"] * rule_s + w["anomaly"] * anomaly_s
    else: w = weights or CONFIG["weights_ml"]; r = w["rule"] * rule_s + w["anomaly"] * anomaly_s + w["ml"] * ml_p
    return round(float(min(100, max(0, r))), 1)

def classify(risk):   # thresholds are PROJECT ASSUMPTIONS - calibrate on real data in a real SOC
    band = "NORMAL" if risk <= 20 else "LOW RISK" if risk <= 40 else "SUSPICIOUS" if risk <= 60 else "HIGH RISK" if risk <= 80 else "CRITICAL INVESTIGATION"
    return ("NORMAL" if risk <= 20 else "SUSPICIOUS" if risk <= 60 else "POTENTIAL INTRUSION"), band
def severity_of(risk): return "CRITICAL" if risk >= 81 else "HIGH" if risk >= 61 else "MEDIUM" if risk >= 41 else "LOW" if risk >= 21 else "INFO"

GENERAL_STEPS = ["Review related logs for the same source IP around this time.", "Check whether the source is a known / authorized host or scanner.",
                 "Compare behaviour with its historical baseline.", "Check authentication logs for related failures or successes.",
                 "Review firewall records for allowed / denied connections.", "Review endpoint telemetry where authorized.",
                 "Determine whether the activity is expected (backup, scan window, deployment)."]
SPECIFIC = {"IDS-001": "Check whether a batch job, load test or misconfigured client explains the connection burst.",
            "IDS-002": "Look for matching failed-login events on the destination service.",
            "IDS-003": "Check whether an approved vulnerability scan was scheduled for this source.",
            "IDS-004": "Compare SYN vs. completed-handshake counts on the destination's firewall.",
            "IDS-005": "Confirm whether a service is legitimately listening on this port.",
            "IDS-006": "Check for scheduled backups / transfers, then verify the destination is approved."}

def explain(matches, an, ml, risk):
    why = " ".join(m["description"] for m in matches) or "No signature matched; risk driven by statistical/ML deviation."
    s = f"{why} Anomaly {an:.0f}/100."
    if ml: s += f" ML probability {ml['rf']:.0f}%."
    return s + f" Hybrid risk {risk:.0f}/100 - needs analyst review, not proof of malicious activity."

def generate_alert(flow_id, flow, res, seq):
    m = sorted(res["matches"], key=lambda x: (-SEV_SCORE[x["severity"]], x["rule_id"]))
    rid, typ = (m[0]["rule_id"], m[0]["name"]) if m else ("HYB-001", "Hybrid Statistical Anomaly")
    return dict(alert_id=f"ALT-{10000 + seq}", flow_id=flow_id, timestamp=str(flow["timestamp"]), source_ip=flow["source_ip"],
                destination_ip=flow["destination_ip"], protocol=str(flow["protocol"]).upper(), source_port=int(flow["source_port"]),
                destination_port=int(flow["destination_port"]), rule_id=rid, alert_type=typ, severity=res["severity"],
                risk_score=res["risk"], rule_score=res["rule_score"], anomaly_score=res["anomaly"],
                ml_score=res["ml"]["rf"] if res["ml"] else None, description=res["reason"], status="NEW",
                matched_rules=json.dumps([x["rule_id"] for x in m]))

# ------------------- 7. DATABASE + SERVICE LAYER (REST-style) --------
SCHEMA = """
CREATE TABLE IF NOT EXISTS network_flows(flow_id TEXT PRIMARY KEY, timestamp TEXT, source_ip TEXT, destination_ip TEXT,
  source_port INT, destination_port INT, protocol TEXT, packet_count REAL, byte_count REAL, duration REAL, connection_count REAL,
  failed_connection_count REAL, syn_count REAL, rst_count REAL, risk_score REAL, classification TEXT, scenario_type TEXT);
CREATE TABLE IF NOT EXISTS alerts(alert_id TEXT PRIMARY KEY, flow_id TEXT REFERENCES network_flows(flow_id), rule_id TEXT,
  alert_type TEXT, severity TEXT, description TEXT, risk_score REAL, rule_score REAL, anomaly_score REAL, ml_score REAL,
  status TEXT DEFAULT 'NEW', created_at TEXT, investigated_at TEXT, resolved_at TEXT, resolution_notes TEXT, matched_rules TEXT);
CREATE TABLE IF NOT EXISTS rules(rule_id TEXT PRIMARY KEY, rule_name TEXT, description TEXT, severity TEXT, threshold TEXT, enabled INT);
CREATE TABLE IF NOT EXISTS incident_notes(note_id INTEGER PRIMARY KEY AUTOINCREMENT, alert_id TEXT REFERENCES alerts(alert_id),
  note TEXT, note_type TEXT, created_at TEXT);
CREATE TABLE IF NOT EXISTS model_results(result_id INTEGER PRIMARY KEY AUTOINCREMENT, flow_id TEXT REFERENCES network_flows(flow_id),
  model_name TEXT, prediction INT, score REAL);
CREATE INDEX IF NOT EXISTS ix_flows_ts ON network_flows(timestamp);  CREATE INDEX IF NOT EXISTS ix_flows_src ON network_flows(source_ip);
CREATE INDEX IF NOT EXISTS ix_alerts_flow ON alerts(flow_id);        CREATE INDEX IF NOT EXISTS ix_alerts_sev ON alerts(severity, status);
CREATE INDEX IF NOT EXISTS ix_notes_alert ON incident_notes(alert_id); CREATE INDEX IF NOT EXISTS ix_results_flow ON model_results(flow_id);
"""
now_s = lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S")

class IDS:
    """Flow collector + IDS engine + alert engine + event store. Methods mirror the REST endpoints
    (ingest = POST /api/flows, set_status = PUT /api/alerts/{id}/status, add_note = POST /api/alerts/{id}/notes)."""
    def __init__(self, db=":memory:", use_ml=True):
        if db != ":memory:" and os.path.exists(db): os.remove(db)
        self.conn = sqlite3.connect(db); self.conn.row_factory = sqlite3.Row
        self.conn.execute("PRAGMA synchronous=OFF"); self.conn.execute("PRAGMA journal_mode=MEMORY")
        self.conn.executescript(SCHEMA); self.tracker = WindowTracker(CONFIG["window_seconds"]); self.seq = 0
        self.use_ml = use_ml and RF is not None
        for i, n, s, d, _, th in RULE_DEFS:
            self.conn.execute("INSERT OR REPLACE INTO rules VALUES(?,?,?,?,?,?)", (i, n, d, s, th.format(**CONFIG["thresholds"]), 1))
        self.conn.commit()

    def score_flow(self, flow):
        f = extract_network_features(flow, self.tracker)
        if not f["valid"]: return dict(valid=False, errors=f["errors"])
        matches = analyze_flow(f); rs = rule_risk(matches); an = calculate_anomaly_score(f)
        ml = ml_scores(f) if self.use_ml else None
        risk = calculate_risk_score(rs, an, ml["rf"] if ml else None); label, band = classify(risk)
        return dict(valid=True, feat=f, matches=matches, rule_score=rs, anomaly=an, ml=ml, risk=risk, label=label, band=band,
                    severity=severity_of(risk), alert=risk >= CONFIG["alert_min_risk"] or bool(matches),
                    reason=explain(matches, an, ml, risk))

    def ingest(self, payload):                                     # POST /api/flows
        if not isinstance(payload, dict): return 400, {"error": "JSON object required"}
        fid = str(payload.get("flow_id") or f"FLOW-{uuid.uuid4().hex[:8]}")
        if self.conn.execute("SELECT 1 FROM network_flows WHERE flow_id=?", (fid,)).fetchone():
            return 409, {"error": "duplicate flow_id", "flow_id": fid}
        flow = {**payload, "flow_id": fid}; res = self.score_flow(flow)
        if not res["valid"]: return 400, {"error": "validation failed", "details": res["errors"]}
        f, c = res["feat"], self.conn
        c.execute("INSERT INTO network_flows VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                  (fid, str(flow["timestamp"]), flow["source_ip"], flow["destination_ip"], int(flow["source_port"]),
                   int(flow["destination_port"]), str(flow["protocol"]).upper(), f["packet_count"], f["byte_count"], f["duration_seconds"],
                   f["connection_count"], f["failed_connection_count"], f["syn_count"], f["rst_count"], res["risk"], res["label"],
                   flow.get("scenario_type", "UNKNOWN")))
        if res["ml"]:
            c.execute("INSERT INTO model_results(flow_id,model_name,prediction,score) VALUES(?,?,?,?)", (fid, "RandomForest", int(res["ml"]["rf"] >= 50), res["ml"]["rf"]))
            c.execute("INSERT INTO model_results(flow_id,model_name,prediction,score) VALUES(?,?,?,?)", (fid, "IsolationForest", res["ml"]["iso"], res["ml"]["iso_raw"]))
        aid = None
        if res["alert"]:
            self.seq += 1; a = generate_alert(fid, flow, res, self.seq); aid = a["alert_id"]
            c.execute("""INSERT INTO alerts(alert_id,flow_id,rule_id,alert_type,severity,description,risk_score,rule_score,anomaly_score,
                         ml_score,status,created_at,matched_rules) VALUES(:alert_id,:flow_id,:rule_id,:alert_type,:severity,:description,
                         :risk_score,:rule_score,:anomaly_score,:ml_score,:status,:timestamp,:matched_rules)""", a)
        c.commit()
        return 201, {"flow_id": fid, "risk_score": res["risk"], "classification": res["label"], "alert_id": aid, "severity": res["severity"] if aid else None}

    def _note(self, aid, text, kind): self.conn.execute("INSERT INTO incident_notes(alert_id,note,note_type,created_at) VALUES(?,?,?,?)", (aid, text, kind, now_s()))

    def set_status(self, aid, new, note=None):                    # PUT /api/alerts/{id}/status
        row = self.conn.execute("SELECT status FROM alerts WHERE alert_id=?", (aid,)).fetchone()
        if not row: return 404, {"error": "alert not found"}
        new = str(new).upper(); old = row["status"]
        if new not in STATUSES: return 400, {"error": "invalid status"}
        if new not in TRANSITIONS[old]: return 409, {"error": f"illegal transition {old} -> {new}"}
        if new == "INVESTIGATING": self.conn.execute("UPDATE alerts SET status=?, investigated_at=? WHERE alert_id=?", (new, now_s(), aid))
        else: self.conn.execute("UPDATE alerts SET status=?, resolved_at=?, resolution_notes=? WHERE alert_id=?", (new, now_s(), note, aid))
        self._note(aid, f"{old} -> {new}", "STATUS")
        if note: self._note(aid, note, "RESOLUTION" if new != "INVESTIGATING" else "ANALYST")
        self.conn.commit(); return 200, {"alert_id": aid, "status": new}

    def add_note(self, aid, note):                                # POST /api/alerts/{id}/notes
        if not self.conn.execute("SELECT 1 FROM alerts WHERE alert_id=?", (aid,)).fetchone(): return 404, {"error": "alert not found"}
        if not isinstance(note, str) or not note.strip() or len(note) > 2000: return 400, {"error": "note must be 1-2000 chars"}
        self._note(aid, note.strip(), "ANALYST"); self.conn.commit(); return 201, {"alert_id": aid}

    def stats(self):                                              # GET /api/dashboard/stats
        q = lambda s: self.conn.execute(s).fetchone()[0] or 0
        return dict(total_flows=q("SELECT COUNT(*) FROM network_flows"), normal=q("SELECT COUNT(*) FROM network_flows WHERE classification='NORMAL'"),
                    suspicious=q("SELECT COUNT(*) FROM network_flows WHERE classification!='NORMAL'"),
                    open_alerts=q("SELECT COUNT(*) FROM alerts WHERE status IN ('NEW','INVESTIGATING')"),
                    critical=q("SELECT COUNT(*) FROM alerts WHERE severity='CRITICAL'"), avg_risk=round(q("SELECT AVG(risk_score) FROM alerts"), 1))

# ------------------- 8. ALERT CORRELATION ----------------------------
def correlate_alerts(df, window=60):
    """EVENT = one flow; ALERT = detection needing attention; INCIDENT = related alerts (same source + type within `window` s)."""
    cols = ["incident_id", "source_ip", "alert_type", "alert_count", "first_seen", "last_seen", "max_severity", "max_risk", "alert_ids"]
    if df is None or df.empty: return pd.DataFrame(columns=cols)
    df = df.assign(created_at=pd.to_datetime(df.created_at)).sort_values("created_at"); inc, live = [], {}
    for r in df.itertuples():
        k = (r.source_ip, r.alert_type); cur = live.get(k)
        if cur and (r.created_at - cur["last_seen"]).total_seconds() <= window:
            cur["alert_count"] += 1; cur["last_seen"] = r.created_at; cur["alert_ids"].append(r.alert_id)
            cur["max_risk"] = max(cur["max_risk"], r.risk_score)
            if SEV_ORDER.index(r.severity) > SEV_ORDER.index(cur["max_severity"]): cur["max_severity"] = r.severity
        else:
            cur = dict(source_ip=r.source_ip, alert_type=r.alert_type, alert_count=1, first_seen=r.created_at, last_seen=r.created_at,
                       max_severity=r.severity, max_risk=r.risk_score, alert_ids=[r.alert_id]); live[k] = cur; inc.append(cur)
    out = pd.DataFrame(inc).sort_values(["max_risk", "alert_count"], ascending=False).reset_index(drop=True)
    out.insert(0, "incident_id", [f"INC-{i + 1:04d}" for i in range(len(out))]); return out[cols]

def alerts_df(ids): return pd.read_sql_query("SELECT a.alert_id,a.alert_type,a.severity,a.risk_score,a.created_at,f.source_ip FROM alerts a JOIN network_flows f USING(flow_id)", ids.conn)

# ------------------- 9. DASHBOARD DATA + HTML ------------------------
def build_dashboard_data(ids):
    c = ids.conn; rows = lambda s, *a: [dict(r) for r in c.execute(s, a).fetchall()]
    fl = pd.read_sql_query("SELECT timestamp,protocol,destination_port,classification FROM network_flows", c)
    if len(fl):
        fl["m"] = pd.to_datetime(fl.timestamp).dt.floor("min"); fl["n"] = fl.classification.eq("NORMAL")
        g = fl.groupby(["m", "n"]).size().unstack(fill_value=0).reindex(columns=[True, False], fill_value=0)
        tl = dict(labels=[t.strftime("%H:%M") for t in g.index], normal=g[True].tolist(), susp=g[False].tolist(), ma=moving_average((g[True] + g[False]).tolist()))
        proto = fl.protocol.value_counts(); ports = fl.destination_port.value_counts().head(10)
    else: tl = dict(labels=[], normal=[], susp=[], ma=[]); proto = ports = pd.Series(dtype=int)
    sev = {r["severity"]: r["n"] for r in rows("SELECT severity, COUNT(*) n FROM alerts GROUP BY severity")}
    types = rows("SELECT alert_type t, COUNT(*) n FROM alerts GROUP BY alert_type ORDER BY n DESC LIMIT 7")
    srcs = rows("SELECT f.source_ip s, COUNT(*) n FROM alerts a JOIN network_flows f USING(flow_id) GROUP BY f.source_ip ORDER BY n DESC LIMIT 8")
    hist = np.histogram([r["risk_score"] for r in rows("SELECT risk_score FROM alerts")] or [0], bins=10, range=(0, 100))[0].tolist()
    al = rows("""SELECT a.*, f.source_ip, f.destination_ip, f.source_port, f.destination_port, f.protocol, f.packet_count, f.byte_count,
                 f.duration, f.connection_count, f.failed_connection_count, f.syn_count, f.rst_count
                 FROM alerts a JOIN network_flows f USING(flow_id) ORDER BY a.created_at DESC, a.alert_id DESC LIMIT 300""")
    notes = defaultdict(list)
    for n in rows("SELECT alert_id, note, note_type, created_at FROM incident_notes ORDER BY note_id"): notes[n["alert_id"]].append(n)
    alerts = [dict(id=a["alert_id"], ts=a["created_at"], src=a["source_ip"], dst=a["destination_ip"], sp=a["source_port"], dp=a["destination_port"],
                   proto=a["protocol"], type=a["alert_type"], rule=a["rule_id"], sev=a["severity"], risk=a["risk_score"], status=a["status"], desc=a["description"],
                   rs=a["rule_score"], an=a["anomaly_score"], ml=a["ml_score"], matched=a["matched_rules"],
                   stats=dict(packets=a["packet_count"], bytes=a["byte_count"], duration=a["duration"], conns=a["connection_count"],
                              failed=a["failed_connection_count"], syn=a["syn_count"], rst=a["rst_count"]),
                   steps=[SPECIFIC[r] for r in json.loads(a["matched_rules"] or "[]") if r in SPECIFIC] + GENERAL_STEPS,
                   notes=notes.get(a["alert_id"], [])) for a in al]
    inc = correlate_alerts(alerts_df(ids), CONFIG["corr_window"]).head(10)
    incidents = [dict(id=r.incident_id, src=r.source_ip, type=r.alert_type, n=int(r.alert_count), first=r.first_seen.strftime("%H:%M:%S"),
                      last=r.last_seen.strftime("%H:%M:%S"), sev=r.max_severity, risk=float(r.max_risk)) for r in inc.itertuples()]
    return dict(stats=ids.stats(), timeline=tl, sev=[int(sev.get(s, 0)) for s in SEV_ORDER], types=types, srcs=srcs, hist=hist,
                proto=dict(labels=proto.index.tolist(), values=proto.tolist()),
                ports=dict(labels=[PORT_NAMES.get(int(p), str(p)) for p in ports.index], values=ports.tolist()),
                alerts=alerts, incidents=incidents, models=MODEL_INFO, weights=CONFIG["weights_ml"] if ids.use_ml else CONFIG["weights_no_ml"],
                generated=now_s(), ml=ids.use_ml)

HTML_TPL = r'''<!DOCTYPE html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<style>
:root{--bg:#0A1020;--panel:#111A2F;--panel2:#16213A;--line:#24324F;--tx:#EAF0FA;--mut:#A5B4CF;--cy:#22D3EE;--ind:#818CF8;--grn:#34D399;--amb:#FBBF24;--org:#FB923C;--red:#F43F5E;--blu:#60A5FA}
*{box-sizing:border-box}body{margin:0;background:radial-gradient(1100px 520px at 8% -10%,#18305c 0%,transparent 60%),radial-gradient(900px 480px at 100% 0,#2a1f55 0%,transparent 55%),var(--bg);color:var(--tx);font:14px/1.5 Inter,system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
.wrap{max-width:1280px;margin:0 auto;padding:26px 22px 40px}header{display:flex;justify-content:space-between;align-items:center;gap:14px;flex-wrap:wrap;margin-bottom:22px}
h1{font-size:25px;margin:0;letter-spacing:-.4px;font-weight:700}h1 span{background:linear-gradient(90deg,var(--cy),var(--ind));-webkit-background-clip:text;background-clip:text;color:transparent}
.sub{color:var(--mut);font-size:13px;margin-top:2px}.pills{display:flex;gap:8px;flex-wrap:wrap}.pill{border:1px solid var(--line);background:#0e1730;color:var(--mut);padding:6px 12px;border-radius:999px;font-size:12px}
.pill b{color:var(--tx)}.dot{display:inline-block;width:8px;height:8px;border-radius:50%;background:var(--grn);margin-right:7px;box-shadow:0 0 10px var(--grn)}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:14px;margin-bottom:16px}
.kpi{background:linear-gradient(160deg,var(--panel2),var(--panel));border:1px solid var(--line);border-radius:14px;padding:16px 16px 14px;position:relative;overflow:hidden}
.kpi:before{content:"";position:absolute;left:0;top:0;width:100%;height:3px;background:var(--c)}.kpi .l{color:var(--mut);font-size:11.5px;text-transform:uppercase;letter-spacing:.09em;font-weight:600}
.kpi .v{font-size:31px;font-weight:700;margin-top:4px;color:var(--tx)}.kpi .v small{font-size:13px;color:var(--mut);font-weight:500}
.grid{display:grid;grid-template-columns:repeat(12,1fr);gap:14px;margin-bottom:14px}.card{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:16px;min-width:0}
.card h3{margin:0 0 12px;font-size:12px;color:var(--mut);font-weight:600;text-transform:uppercase;letter-spacing:.09em}.c4{grid-column:span 4}.c6{grid-column:span 6}.c8{grid-column:span 8}.c12{grid-column:span 12}
.ch{position:relative;height:250px}@media(max-width:920px){.c4,.c6,.c8{grid-column:span 12}}
.filters{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:12px}.filters label{font-size:11px;color:var(--mut);display:flex;flex-direction:column;gap:3px;text-transform:uppercase;letter-spacing:.07em}
select{background:#0c1429;color:var(--tx);border:1px solid var(--line);border-radius:8px;padding:7px 10px;font:inherit;min-width:135px}select:focus{outline:2px solid var(--cy)}
.tw{overflow-x:auto;max-height:430px;overflow-y:auto;border-radius:10px;border:1px solid var(--line)}table{width:100%;border-collapse:collapse;font-size:13px}
th{position:sticky;top:0;background:#0d1730;color:var(--mut);text-align:left;padding:10px 12px;font-size:11px;text-transform:uppercase;letter-spacing:.08em;white-space:nowrap}
td{padding:9px 12px;border-top:1px solid #1b2846;white-space:nowrap;color:var(--tx)}tbody tr:hover{background:#16233f}tr.clk{cursor:pointer}.mono{font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12.5px}
.b{padding:2px 9px;border-radius:6px;font-size:11px;font-weight:700;letter-spacing:.04em;border:1px solid}
.b-INFO{color:#93C5FD;background:#60A5FA22;border-color:#60A5FA66}.b-LOW{color:#6EE7B7;background:#34D39922;border-color:#34D39966}.b-MEDIUM{color:#FCD34D;background:#FBBF2422;border-color:#FBBF2466}
.b-HIGH{color:#FDBA74;background:#FB923C22;border-color:#FB923C77}.b-CRITICAL{color:#FDA4AF;background:#F43F5E26;border-color:#F43F5E88}
.st{padding:2px 9px;border-radius:999px;font-size:11px;font-weight:600;border:1px solid var(--line)}.st-NEW{color:#67E8F9;background:#22D3EE1f}.st-INVESTIGATING{color:#C4B5FD;background:#818CF822}.st-RESOLVED{color:#6EE7B7;background:#34D39920}.st-FALSE_POSITIVE{color:#CBD5E1;background:#94A3B825}
.rk{display:inline-block;min-width:34px;text-align:center;font-weight:700}.mut{color:var(--mut)}.num{text-align:right}
#modal{position:fixed;inset:0;background:#030712cc;display:none;align-items:center;justify-content:center;padding:18px;z-index:9;backdrop-filter:blur(3px)}
#modal .box{background:var(--panel);border:1px solid var(--line);border-radius:16px;max-width:860px;width:100%;max-height:90vh;overflow:auto;padding:22px}
.mg{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px;margin:14px 0}.mg div{background:#0d1730;border:1px solid var(--line);border-radius:10px;padding:9px 12px}
.mg small{display:block;color:var(--mut);font-size:11px;text-transform:uppercase;letter-spacing:.07em}.x{float:right;background:#0d1730;color:var(--tx);border:1px solid var(--line);border-radius:8px;padding:5px 12px;cursor:pointer}
.box h2{margin:0 0 4px;font-size:19px}.box h4{margin:16px 0 6px;color:var(--cy);font-size:12px;text-transform:uppercase;letter-spacing:.09em}.box ol,.box ul{margin:0;padding-left:20px;color:var(--tx)}.box li{margin:3px 0}
.foot{color:var(--mut);font-size:12px;text-align:center;margin-top:18px}.empty{color:var(--mut);padding:18px;text-align:center}
</style></head><body><div class="wrap">
<header><div><h1>&#128737; Network <span>IDS</span> &mdash; SOC Console</h1><div class="sub">Hybrid detection: signatures + statistical anomaly + machine learning &middot; synthetic traffic only</div></div>
<div class="pills" id="pills"></div></header>
<div class="kpis" id="kpis"></div>
<div class="grid"><div class="card c8"><h3>Traffic over time (flows / minute)</h3><div class="ch"><canvas id="c1"></canvas></div></div>
<div class="card c4"><h3>Normal vs suspicious</h3><div class="ch"><canvas id="c2"></canvas></div></div>
<div class="card c4"><h3>Alerts by severity</h3><div class="ch"><canvas id="c3"></canvas></div></div>
<div class="card c4"><h3>Top alert types</h3><div class="ch"><canvas id="c4"></canvas></div></div>
<div class="card c4"><h3>Protocol distribution</h3><div class="ch"><canvas id="c5"></canvas></div></div>
<div class="card c4"><h3>Destination port distribution</h3><div class="ch"><canvas id="c6"></canvas></div></div>
<div class="card c4"><h3>Top source IPs by alerts</h3><div class="ch"><canvas id="c7"></canvas></div></div>
<div class="card c4"><h3>Risk score distribution (alerts)</h3><div class="ch"><canvas id="c8"></canvas></div></div></div>
<div class="card" style="margin-bottom:14px"><h3>Recent alerts &middot; click a row to investigate</h3>
<div class="filters"><label>Severity<select id="f_sev"></select></label><label>Protocol<select id="f_proto"></select></label><label>Alert type<select id="f_type"></select></label>
<label>Status<select id="f_status"></select></label><label>Time range<select id="f_time"><option value="0">All time</option><option value="5">Last 5 min</option><option value="15">Last 15 min</option><option value="30">Last 30 min</option></select></label></div>
<div class="tw"><table><thead><tr><th>Time</th><th>Source IP</th><th>Destination IP</th><th>Proto</th><th>Alert</th><th>Severity</th><th>Risk</th><th>Status</th></tr></thead><tbody id="tb"></tbody></table></div>
<div class="mut" id="cnt" style="margin-top:8px;font-size:12px"></div></div>
<div class="grid"><div class="card c6"><h3>Correlated incidents (same source + type within 60 s)</h3><div class="tw"><table><thead><tr><th>ID</th><th>Source</th><th>Type</th><th class="num">Alerts</th><th>Window</th><th>Severity</th></tr></thead><tbody id="inc"></tbody></table></div></div>
<div class="card c6"><h3>Model evaluation &middot; held-out test set</h3><div class="tw"><table><thead><tr><th>Detector</th><th class="num">Acc</th><th class="num">Prec</th><th class="num">Recall</th><th class="num">F1</th></tr></thead><tbody id="mod"></tbody></table></div>
<div id="cm" style="margin-top:12px"></div></div></div>
<div class="foot">Defensive education project &mdash; all suspicious behaviour is represented as synthetic flow records. Risk scores and thresholds are project assumptions; a match means &ldquo;investigate&rdquo;, not &ldquo;proven attack&rdquo;.</div>
</div><div id="modal"><div class="box" id="mbox"></div></div>
<script>
const D=__DATA__;const $=id=>document.getElementById(id);const esc=s=>String(s==null?'':s).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const SEV=['INFO','LOW','MEDIUM','HIGH','CRITICAL'],SC={INFO:'#60A5FA',LOW:'#34D399',MEDIUM:'#FBBF24',HIGH:'#FB923C',CRITICAL:'#F43F5E'};
const badge=s=>`<span class="b b-${esc(s)}">${esc(s)}</span>`,pill=s=>`<span class="st st-${esc(s)}">${esc(s).replace('_',' ')}</span>`;
const rcol=r=>r>=81?SC.CRITICAL:r>=61?SC.HIGH:r>=41?SC.MEDIUM:r>=21?SC.LOW:SC.INFO,fmt=n=>Number(n).toLocaleString();
const S=D.stats;$('pills').innerHTML=`<span class="pill"><span class="dot"></span><b>Live simulation</b></span><span class="pill">Weights <b>${Object.entries(D.weights).map(([k,v])=>k+' '+Math.round(v*100)+'%').join(' &middot; ')}</b></span><span class="pill">ML <b>${D.ml?'enabled':'disabled'}</b></span><span class="pill">Updated <b>${esc(D.generated)}</b></span>`;
$('kpis').innerHTML=[['Total network flows',fmt(S.total_flows),'#22D3EE'],['Normal traffic',fmt(S.normal),'#34D399'],['Suspicious traffic',fmt(S.suspicious),'#FB923C'],['Open alerts',fmt(S.open_alerts),'#818CF8'],['Critical alerts',fmt(S.critical),'#F43F5E'],['Avg risk score',S.avg_risk+' <small>/ 100</small>','#FBBF24']].map(k=>`<div class="kpi" style="--c:${k[2]}"><div class="l">${k[0]}</div><div class="v">${k[1]}</div></div>`).join('');
if(window.Chart){Chart.defaults.color='#A5B4CF';Chart.defaults.borderColor='#1f2c49';Chart.defaults.font.family='Inter,system-ui,sans-serif';
const P=['#22D3EE','#818CF8','#34D399','#FBBF24','#FB923C','#F43F5E','#60A5FA','#C084FC','#2DD4BF','#F472B6'];
const sc=h=>({x:{grid:{display:false},ticks:{maxRotation:0,autoSkip:true}},y:{beginAtZero:true,grid:{color:'#1c2946'},ticks:{precision:0}}});
const hs=()=>({x:{beginAtZero:true,grid:{color:'#1c2946'},ticks:{precision:0}},y:{grid:{display:false}}});
const mk=(id,type,labels,sets,o={})=>new Chart($(id),{type,data:{labels,datasets:sets},options:Object.assign({responsive:true,maintainAspectRatio:false,plugins:{legend:{display:sets.length>1||['doughnut'].includes(type),position:'bottom',labels:{boxWidth:10,usePointStyle:true,padding:14}}}},o)});
const T=D.timeline;mk('c1','line',T.labels,[{label:'Normal',data:T.normal,borderColor:'#22D3EE',backgroundColor:'#22D3EE22',fill:true,tension:.35,pointRadius:0,borderWidth:2},{label:'Suspicious',data:T.susp,borderColor:'#F43F5E',backgroundColor:'#F43F5E22',fill:true,tension:.35,pointRadius:0,borderWidth:2},{label:'3-min moving avg (all)',data:T.ma,borderColor:'#818CF8',borderDash:[5,4],tension:.35,pointRadius:0,borderWidth:1.6}],{scales:sc(),interaction:{mode:'index',intersect:false}});
mk('c2','doughnut',['Normal','Suspicious / intrusion'],[{data:[S.normal,S.suspicious],backgroundColor:['#22D3EE','#F43F5E'],borderColor:'#111A2F',borderWidth:3}],{cutout:'68%'});
mk('c3','bar',SEV,[{data:D.sev,backgroundColor:SEV.map(s=>SC[s]),borderRadius:6}],{scales:sc()});
mk('c4','bar',D.types.map(t=>t.t),[{data:D.types.map(t=>t.n),backgroundColor:'#818CF8',borderRadius:6}],{indexAxis:'y',scales:hs()});
mk('c5','doughnut',D.proto.labels,[{data:D.proto.values,backgroundColor:P,borderColor:'#111A2F',borderWidth:3}],{cutout:'62%'});
mk('c6','bar',D.ports.labels,[{data:D.ports.values,backgroundColor:'#22D3EE',borderRadius:6}],{scales:sc()});
mk('c7','bar',D.srcs.map(t=>t.s),[{data:D.srcs.map(t=>t.n),backgroundColor:'#FB923C',borderRadius:6}],{indexAxis:'y',scales:hs()});
mk('c8','bar',D.hist.map((_,i)=>i*10+'-'+(i*10+10)),[{data:D.hist,backgroundColor:D.hist.map((_,i)=>rcol(i*10+5)),borderRadius:6}],{scales:sc()});}
else document.querySelectorAll('.ch').forEach(e=>e.innerHTML='<div class="empty">Charts need internet access to load Chart.js.</div>');
const uniq=k=>[...new Set(D.alerts.map(a=>a[k]))].sort();const fill=(id,l,vals)=>$(id).innerHTML=`<option value="">${l}</option>`+vals.map(v=>`<option>${esc(v)}</option>`).join('');
fill('f_sev','All severities',SEV.slice().reverse());fill('f_proto','All protocols',uniq('proto'));fill('f_type','All alert types',uniq('type'));fill('f_status','All statuses',['NEW','INVESTIGATING','RESOLVED','FALSE_POSITIVE']);
const tms=a=>new Date(a.ts.replace(' ','T')).getTime(),ref=D.alerts.length?Math.max(...D.alerts.map(tms)):0;
function render(){const f={sev:$('f_sev').value,proto:$('f_proto').value,type:$('f_type').value,status:$('f_status').value},w=+$('f_time').value;
const rows=D.alerts.filter(a=>(!f.sev||a.sev===f.sev)&&(!f.proto||a.proto===f.proto)&&(!f.type||a.type===f.type)&&(!f.status||a.status===f.status)&&(!w||ref-tms(a)<=w*60000));
$('tb').innerHTML=rows.length?rows.slice(0,150).map(a=>`<tr class="clk" data-id="${esc(a.id)}"><td class="mono">${esc(a.ts.slice(11))}</td><td class="mono">${esc(a.src)}</td><td class="mono">${esc(a.dst)}</td><td>${esc(a.proto)}</td><td>${esc(a.type)}</td><td>${badge(a.sev)}</td><td><span class="rk" style="color:${rcol(a.risk)}">${Math.round(a.risk)}</span></td><td>${pill(a.status)}</td></tr>`).join(''):'<tr><td colspan="8" class="empty">No alerts match these filters.</td></tr>';
$('cnt').textContent=`Showing ${Math.min(rows.length,150)} of ${rows.length} matching alerts (latest ${D.alerts.length} loaded).`;}
['f_sev','f_proto','f_type','f_status','f_time'].forEach(i=>$(i).onchange=render);render();
$('tb').onclick=e=>{const tr=e.target.closest('tr[data-id]');if(!tr)return;const a=D.alerts.find(x=>x.id===tr.dataset.id),s=a.stats;
$('mbox').innerHTML=`<button class="x" onclick="$('modal').style.display='none'">Close</button><h2>${esc(a.id)} &middot; ${esc(a.type)}</h2><div>${badge(a.sev)} ${pill(a.status)} <span class="mut mono">${esc(a.ts)}</span></div>
<div class="mg"><div><small>Source</small><span class="mono">${esc(a.src)}:${esc(a.sp)}</span></div><div><small>Destination</small><span class="mono">${esc(a.dst)}:${esc(a.dp)}</span></div><div><small>Protocol</small>${esc(a.proto)}</div><div><small>Matched rules</small>${esc(JSON.parse(a.matched||'[]').join(', ')||a.rule)}</div></div>
<div class="mg"><div><small>Rule score</small>${a.rs}</div><div><small>Anomaly score</small>${a.an}</div><div><small>ML probability</small>${a.ml==null?'n/a':a.ml.toFixed(1)+'%'}</div><div><small>Final risk</small><b style="color:${rcol(a.risk)}">${a.risk}/100</b></div></div>
<div class="mg"><div><small>Packets</small>${fmt(s.packets)}</div><div><small>Bytes</small>${fmt(s.bytes)}</div><div><small>Duration</small>${s.duration}s</div><div><small>Connections / failed</small>${s.conns} / ${s.failed}</div><div><small>SYN / RST</small>${s.syn} / ${s.rst}</div></div>
<h4>Reason</h4><div>${esc(a.desc)}</div><h4>Recommended investigation steps</h4><ol>${a.steps.map(x=>`<li>${esc(x)}</li>`).join('')}</ol>
<h4>Incident timeline &amp; analyst notes</h4>${a.notes.length?`<ul>${a.notes.map(n=>`<li><span class="mut mono">${esc(n.created_at)}</span> &middot; <b>${esc(n.note_type)}</b> &middot; ${esc(n.note)}</li>`).join('')}</ul>`:'<div class="mut">No notes yet. Use the analyst panel below the dashboard to update status and add notes.</div>'}`;$('modal').style.display='flex';};
$('modal').onclick=e=>{if(e.target.id==='modal')$('modal').style.display='none'};
$('inc').innerHTML=D.incidents.map(i=>`<tr><td class="mono">${esc(i.id)}</td><td class="mono">${esc(i.src)}</td><td>${esc(i.type)}</td><td class="num"><b>${i.n}</b></td><td class="mono">${esc(i.first)}&ndash;${esc(i.last)}</td><td>${badge(i.sev)}</td></tr>`).join('')||'<tr><td colspan="6" class="empty">No incidents.</td></tr>';
const M=D.models;$('mod').innerHTML=(M.models||[]).map(m=>`<tr><td>${esc(m.model)}</td><td class="num">${m.accuracy}</td><td class="num">${m.precision}</td><td class="num">${m.recall}</td><td class="num">${m.f1}</td></tr>`).join('')+'<tr><td colspan="5" class="mut" style="font-size:11px;text-transform:uppercase;letter-spacing:.08em;background:#0d1730">Detection layers compared</td></tr>'+(M.compare||[]).map(m=>`<tr><td>${esc(m.model)}</td><td class="num">${m.accuracy}</td><td class="num">${m.precision}</td><td class="num">${m.recall}</td><td class="num">${m.f1}</td></tr>`).join('');
if(M.cm){const c=M.cm;$('cm').innerHTML=`<div class="mut" style="font-size:12px;margin-bottom:6px">Random Forest confusion matrix (test n=${M.n_test})</div><div style="display:grid;grid-template-columns:auto 1fr 1fr;gap:6px;font-size:13px;max-width:420px"><span></span><span class="mut">Pred NORMAL</span><span class="mut">Pred SUSPICIOUS</span><span class="mut">Actual NORMAL</span><div class="mg" style="margin:0;display:block;border-color:#34D39988"><small>True negative</small><b>${c[0][0]}</b></div><div class="mg" style="margin:0;display:block;border-color:#FB923C88"><small>False positive</small><b>${c[0][1]}</b></div><span class="mut">Actual SUSP.</span><div class="mg" style="margin:0;display:block;border-color:#F43F5E88"><small>False negative</small><b>${c[1][0]}</b></div><div class="mg" style="margin:0;display:block;border-color:#22D3EE88"><small>True positive</small><b>${c[1][1]}</b></div></div>`}
</script></body></html>'''

def render_dashboard(ids, path="ids_dashboard.html"):
    html = HTML_TPL.replace("__DATA__", json.dumps(build_dashboard_data(ids), default=str).replace("<", "\\u003c"))
    os.makedirs("dashboard_site", exist_ok=True)              # ONLY this folder is served by the link (not your DB / data)
    for pth in (path, "dashboard_site/index.html"): open(pth, "w", encoding="utf-8").write(html)
    return html

def serve_dashboard(port=8000):
    """Serve dashboard_site/ on a local port and print a clickable link (Colab proxy). Re-running the cell reuses the port."""
    import threading, functools, http.server, socketserver
    old = globals().get("_DASH_SERVER")
    if old: old.shutdown(); old.server_close()
    class Q(http.server.SimpleHTTPRequestHandler):
        def log_message(self, *a): pass
        def end_headers(self): self.send_header("Cache-Control", "no-store"); super().end_headers()
    socketserver.ThreadingTCPServer.allow_reuse_address = True; srv = None
    for p_ in range(port, port + 10):
        try: srv = socketserver.ThreadingTCPServer(("", p_), functools.partial(Q, directory=os.path.abspath("dashboard_site"))); break
        except OSError: continue
    if srv is None: print("Could not open a local port."); return None
    globals()["_DASH_SERVER"] = srv; threading.Thread(target=srv.serve_forever, daemon=True).start(); port = srv.server_address[1]
    try:
        from google.colab import output
        try:
            url = output.eval_js(f"google.colab.kernel.proxyPort({port}, {{'cache': true}})")
            display(HTML(f"<a href='{url}' target='_blank' style=\"display:inline-block;padding:14px 26px;border-radius:12px;background:linear-gradient(90deg,#22D3EE,#818CF8);"
                         f"color:#0A1020;font:700 16px system-ui;text-decoration:none\">&#128737; Open SOC Dashboard in a new tab &rarr;</a>"
                         f"<div style='font:13px system-ui;margin-top8px;color:#555'>{url}<br>Works while this Colab runtime is running. After analyst updates, click <b>Refresh dashboard</b> below, then reload that tab.</div>".replace("margin-top8px", "margin-top:8px")))
        except Exception:
            output.serve_kernel_port_as_window(port)           # official Colab helper: prints a clickable link
    except ImportError:
        print(f"Not running in Colab - open http://localhost:{port}/ in your browser.")
    return srv

# ------------------- 10. REPORTS / SIEM EXPORT -----------------------
def export_siem(ids, path="reports/siem_alerts.jsonl"):
    os.makedirs("reports", exist_ok=True); n = 0; first = None
    with open(path, "w") as fh:
        for r in ids.conn.execute("SELECT a.alert_id,a.created_at,f.source_ip,f.destination_ip,a.severity,a.alert_type,a.risk_score,a.status FROM alerts a JOIN network_flows f USING(flow_id) ORDER BY a.risk_score DESC"):
            j = dict(alert_id=r[0], timestamp=r[1], source_ip=r[2], destination_ip=r[3], severity=r[4], rule=r[5], risk_score=r[6], status=r[7])
            fh.write(json.dumps(j) + "\n"); first = first or j; n += 1
    return n, first

def generate_incident_report(ids, path="reports/incident_report.md"):
    os.makedirs("reports", exist_ok=True); d = build_dashboard_data(ids); s = d["stats"]
    L = [f"# IDS Incident Report", f"_Generated {d['generated']} from synthetic traffic - defensive education project_", "",
         "## Summary", f"- Flows analysed: **{s['total_flows']}** (normal {s['normal']}, suspicious {s['suspicious']})",
         f"- Alerts: **{sum(d['sev'])}**  | Open: {s['open_alerts']} | Critical: {s['critical']} | Avg risk: {s['avg_risk']}",
         "- Alerts by severity: " + ", ".join(f"{k}={v}" for k, v in zip(SEV_ORDER, d["sev"])), "", "## Top correlated incidents",
         "| Incident | Source | Type | Alerts | Window | Severity |", "|---|---|---|---|---|---|"]
    L += [f"| {i['id']} | {i['src']} | {i['type']} | {i['n']} | {i['first']}-{i['last']} | {i['sev']} |" for i in d["incidents"][:8]]
    L += ["", "## Highest-risk alerts"]
    for a in sorted(d["alerts"], key=lambda x: -x["risk"])[:5]:
        L += [f"### {a['id']} - {a['type']} ({a['sev']}, risk {a['risk']})", f"- {a['src']} -> {a['dst']}:{a['dp']} {a['proto']} at {a['ts']}  | status: {a['status']}",
              f"- Reason: {a['desc']}", "- Notes: " + ("; ".join(f"[{n['note_type']}] {n['note']}" for n in a["notes"]) or "none"), ""]
    L += ["## Caveats", "Scores/thresholds are project assumptions. Synthetic data is easier to separate than real traffic, so model metrics are optimistic."]
    open(path, "w").write("\n".join(L)); return "\n".join(L)

# ------------------- 11. RUN EVERYTHING ------------------------------
def banner(t): print(f"\n{'=' * 8} {t} {'=' * 8}")
banner("STEP 1  Synthetic dataset"); os.makedirs("data", exist_ok=True)
DF = generate_dataset(CONFIG["n_flows"], CONFIG["suspicious_share"], CONFIG["seed"]); DF.to_csv("data/network_traffic.csv", index=False)
print(f"Saved data/network_traffic.csv  ({len(DF):,} rows)  |  labels:", DF.label.value_counts().to_dict())
display(DF.groupby("scenario_type").size().rename("flows").to_frame().T)

banner("STEP 2  Feature engineering + ML training (real metrics)")
FEATS = featurize_df(DF); train_models(FEATS)
print(f"Train {MODEL_INFO['n_train']:,} / Test {MODEL_INFO['n_test']:,} (stratified).  Baseline learned from normal training flows only.")
display(pd.DataFrame(MODEL_INFO["models"]).set_index("model")); display(pd.DataFrame(MODEL_INFO["compare"]).set_index("model"))
cm = MODEL_INFO["cm"]; print(f"Random Forest confusion matrix  [[TN={cm[0][0]}, FP={cm[0][1]}], [FN={cm[1][0]}, TP={cm[1][1]}]]")
print("Note: synthetic data is simpler than real traffic, so treat these scores as optimistic (the rules were written against the same synthetic patterns, so ~100% is expected, not a real-world claim). Accuracy alone is never enough - watch recall AND false positives.")

banner("STEP 3  Live simulation -> IDS -> alerts -> SQLite")
ids = IDS("ids_events.db", use_ml=CONFIG["use_ml"]); shown = 0
for flow in traffic_simulator(CONFIG["live_mode"], CONFIG["live_speed"], CONFIG["live_flows"]):
    code, body = ids.ingest(flow)
    if code == 201 and (body["alert_id"] and shown < 6):
        shown += 1; print(f"  ALERT {body['alert_id']} | {flow['source_ip']:>15} -> {flow['destination_ip']}:{flow['destination_port']:<5} | {body['severity']:<8} risk {body['risk_score']:>5}  ({flow['scenario_type']})")
print("Dashboard stats:", ids.stats())

banner("STEP 4  Analyst workflow (NEW -> INVESTIGATING -> RESOLVED / FALSE_POSITIVE)")
top = [r[0] for r in ids.conn.execute("SELECT alert_id FROM alerts ORDER BY risk_score DESC LIMIT 3")]
if len(top) == 3:
    print(ids.set_status(top[0], "INVESTIGATING")); print(ids.add_note(top[0], "Source not in asset inventory; checking firewall + auth logs."))
    print(ids.set_status(top[1], "INVESTIGATING")); print(ids.set_status(top[1], "RESOLVED", "Confirmed scan pattern; source blocked by network team (ticket SIM-101)."))
    print(ids.set_status(top[2], "FALSE_POSITIVE", "Matches scheduled backup window; baseline to be tuned."))
print("Illegal move NEW -> RESOLVED is rejected:", ids.set_status(next(r[0] for r in ids.conn.execute("SELECT alert_id FROM alerts WHERE status='NEW'")), "RESOLVED"))

banner("STEP 5  Alert correlation")
INC = correlate_alerts(alerts_df(ids), CONFIG["corr_window"])
print(f"{int(ids.conn.execute('SELECT COUNT(*) FROM alerts').fetchone()[0])} alerts  ->  {len(INC)} correlated incidents"); display(INC.head(6))

banner("STEP 6  Safe detection scenarios A-G (records only)")
sr, T0, sc_rows = np.random.default_rng(5), datetime.now(), []
SCEN = [("A Normal HTTPS", "NORMAL_WEB", 443, "No alert"), ("B Normal DNS", "NORMAL_DNS", None, "No alert"), ("C Repeated failures", "REPEATED_FAILED_CONNECTIONS", None, "Repeated Failed Connections"),
        ("D High connection rate", "HIGH_CONNECTION_RATE", None, "High Connection Rate"), ("E Multi-port pattern", MP, None, "Many Destination Ports"),
        ("F SYN-heavy", "SYN_HEAVY_PATTERN", None, "SYN-Heavy Behaviour"), ("G High volume", "HIGH_TRAFFIC_VOLUME", None, "Abnormally High Traffic Volume")]
for name, scn, dp, exp in SCEN:
    tmp = IDS(":memory:", use_ml=CONFIG["use_ml"])
    flows = make_burst(sr, T0, "203.0.113.45", "198.51.100.20", 15) if scn == MP else [make_flow(sr, scn, T0, "203.0.113.45" if scn != "NORMAL_WEB" and scn != "NORMAL_DNS" else "192.0.2.15", "198.51.100.20", dp)]
    rs_ = [tmp.score_flow(f) for f in flows]; r = max(rs_, key=lambda x: x["risk"]); f = r["feat"]
    sc_rows.append(dict(scenario=name, conn_rate=round(f["connection_rate"], 1), fail_ratio=round(f["failure_ratio"], 2), syn=int(f["syn_count"]), uniq_ports=int(max(x["feat"]["unique_destination_ports"] for x in rs_)),
                        rules=", ".join(sorted({m["rule_id"] for x in rs_ for m in x["matches"]})) or "-", anomaly=r["anomaly"], ml_pct=round(r["ml"]["rf"]) if r["ml"] else None,
                        risk=r["risk"], classification=r["label"], expected=exp))
display(pd.DataFrame(sc_rows).set_index("scenario"))

banner("STEP 7  Automated tests (30)")
def bf(**kw):
    d = dict(flow_id="T-1", timestamp="2025-01-01 10:00:00", source_ip="192.0.2.15", destination_ip="198.51.100.20", source_port=49152, destination_port=443,
             protocol="TCP", packet_count=18, byte_count=12400, duration_seconds=2.8, connection_count=3, failed_connection_count=0, syn_count=2, rst_count=0); d.update(kw); return d
fresh = lambda: IDS(":memory:", use_ml=True); run = lambda **kw: fresh().score_flow(bf(**kw)); rules_of = lambda r: [m["rule_id"] for m in r["matches"]]
HC = dict(connection_count=120, duration_seconds=3, syn_count=110, failed_connection_count=5, packet_count=300, byte_count=30000)
FA = dict(connection_count=40, failed_connection_count=34, syn_count=40, rst_count=34, packet_count=100, duration_seconds=10, byte_count=7000)
SY = dict(syn_count=400, packet_count=420, byte_count=25200, duration_seconds=2, connection_count=5)
VO = dict(byte_count=80_000_000, packet_count=60000, duration_seconds=60)
TESTS = []
def tst(i, scen, exp, act, ok): TESTS.append(dict(id=f"T{i:02d}", scenario=scen, expected=exp, actual=str(act)[:48], result="PASS" if ok else "FAIL"))
r = run(); tst(1, "Normal TCP flow", "NORMAL, no rules", f"{r['label']} {rules_of(r)}", r["label"] == "NORMAL" and not r["matches"])
r = run(protocol="UDP", destination_port=123, packet_count=4, byte_count=380, duration_seconds=.2, connection_count=1, syn_count=0); tst(2, "Normal UDP flow", "NORMAL", r["label"], r["label"] == "NORMAL")
r = run(protocol="UDP", destination_port=53, packet_count=3, byte_count=270, duration_seconds=.05, connection_count=1, syn_count=0); tst(3, "Normal DNS flow", "NORMAL", r["label"], r["label"] == "NORMAL")
r = run(); tst(4, "Normal HTTPS flow", "risk <= 20", r["risk"], r["risk"] <= 20)
r = run(**HC); tst(5, "High connection rate", "IDS-001", rules_of(r), "IDS-001" in rules_of(r))
r = run(**FA); tst(6, "Repeated failed connections", "IDS-002", rules_of(r), "IDS-002" in rules_of(r))
t7 = fresh(); r7 = None
for k in range(12): r7 = t7.score_flow(bf(flow_id=f"M{k}", timestamp=f"2025-01-01 10:00:{k:02d}", destination_port=2000 + k, packet_count=3, byte_count=180, duration_seconds=.1, connection_count=1, syn_count=1))
tst(7, "Multi-port pattern (12 ports/60s)", "IDS-003", rules_of(r7), "IDS-003" in rules_of(r7))
r = run(**SY); tst(8, "SYN-heavy pattern", "IDS-004", rules_of(r), "IDS-004" in rules_of(r))
r = run(**VO); tst(9, "High traffic volume", "IDS-006", rules_of(r), "IDS-006" in rules_of(r))
for i, (nm, kw) in enumerate([("Invalid source IP", dict(source_ip="999.1.1.1")), ("Invalid destination IP", dict(destination_ip="not-an-ip")), ("Invalid source port", dict(source_port=70000)),
                              ("Invalid destination port", dict(destination_port=-5)), ("Unsupported protocol", dict(protocol="XYZ"))], start=10):
    r = run(**kw); tst(i, nm, "rejected", r["errors"] if not r["valid"] else "accepted", not r["valid"])
r = run(packet_count=None); tst(15, "Missing packet count", "handled, pps = 0", f"valid={r['valid']} pps={r['feat']['packets_per_second']}", r["valid"] and r["feat"]["packets_per_second"] == 0)
r = run(duration_seconds=0); tst(16, "Zero duration", "finite features", round(r["feat"]["bytes_per_second"], 1), np.isfinite(r["feat"]["bytes_per_second"]))
f17 = extract_network_features(bf()); tst(17, "Feature extraction", "bps=12400/2.8, syn_ratio=2/18", f"{f17['bytes_per_second']:.1f}", abs(f17["bytes_per_second"] - 12400 / 2.8) < 1e-6 and abs(f17["syn_ratio"] - 2 / 18) < 1e-9)
fh = extract_network_features(bf(**HC)); fh.update(_dport=443, _proto="TCP"); tst(18, "Rule detection + configurable threshold", "fires; off when threshold raised", "ok", any(m["rule_id"] == "IDS-001" for m in analyze_flow(fh)) and not any(m["rule_id"] == "IDS-001" for m in analyze_flow(fh, {"conn_rate": 1e6})))
a_n, a_h = calculate_anomaly_score(extract_network_features(bf())), calculate_anomaly_score(fh); tst(19, "Anomaly score", "0-100 and attack > normal", f"{a_n} < {a_h}", 0 <= a_n < a_h <= 100)
tst(20, "Risk score weights", "73.4 (ML) / 72.0 (no ML)", f"{calculate_risk_score(80, 60, 78)} / {calculate_risk_score(80, 60)}", calculate_risk_score(80, 60, 78) == 73.4 and calculate_risk_score(80, 60) == 72.0)
t21 = fresh(); res = t21.score_flow(bf(**HC)); al = generate_alert("T-1", bf(**HC), res, 1); need = {"alert_id", "timestamp", "source_ip", "destination_ip", "protocol", "source_port", "destination_port", "rule_id", "alert_type", "severity", "risk_score", "description", "status"}
tst(21, "Alert creation", "all fields, status NEW", f"{al['alert_id']} {al['status']}", need <= set(al) and al["status"] == "NEW")
cd = pd.DataFrame([dict(alert_id=f"A{i}", source_ip="192.0.2.9", alert_type="X", severity="LOW", risk_score=30, created_at=f"2025-01-01 10:00:{i * 5:02d}") for i in range(5)] +
                  [dict(alert_id="A9", source_ip="192.0.2.9", alert_type="X", severity="HIGH", risk_score=70, created_at="2025-01-01 10:05:00")])
ci = correlate_alerts(cd, 60); tst(22, "Alert correlation", "6 alerts -> 2 incidents", f"{len(ci)} incidents", len(ci) == 2 and sorted(ci.alert_count.tolist()) == [1, 5])
t23 = fresh(); aid = t23.ingest(bf(flow_id="S1", **HC))[1]["alert_id"]; c1_ = t23.set_status(aid, "RESOLVED")[0]; c2_ = t23.set_status(aid, "INVESTIGATING")[0]
tst(23, "Alert status update", "NEW->RESOLVED 409; ->INVESTIGATING 200", f"{c1_}, {c2_}", (c1_, c2_) == (409, 200))
c3_ = t23.add_note(aid, "Checked firewall logs")[0]; n_ = t23.conn.execute("SELECT COUNT(*) FROM incident_notes WHERE alert_id=? AND note_type='ANALYST'", (aid,)).fetchone()[0]; tst(24, "Analyst note", "201 + stored", f"{c3_}, notes={n_}", c3_ == 201 and n_ == 1)
t25 = fresh(); t25.ingest(bf(flow_id="D1")); row = t25.conn.execute("SELECT classification FROM network_flows WHERE flow_id='D1'").fetchone(); tst(25, "Database storage", "row stored", dict(row) if row else None, row is not None)
t26 = fresh(); t26.ingest(bf(flow_id="D1")); t26.ingest(bf(flow_id="D2", **HC)); s_ = t26.stats(); tst(26, "Dashboard statistics", "total=2, suspicious>=1", s_, s_["total_flows"] == 2 and s_["suspicious"] >= 1)
pn, ps = run()["ml"]["rf"], run(**SY)["ml"]["rf"]; tst(27, "ML prediction", "SYN-heavy prob > normal, > 50%", f"{ps:.0f}% vs {pn:.0f}%", ps > pn and ps > 50)
cs = [fresh().ingest(bf(source_ip="bad"))[0], fresh().ingest("not-json")[0]]; tst(28, "API validation", "400, 400", cs, cs == [400, 400])
e_ = fresh(); tst(29, "Empty dataset", "zero stats, no crash", e_.stats()["total_flows"], e_.stats()["total_flows"] == 0 and featurize_df(pd.DataFrame()).empty and correlate_alerts(pd.DataFrame()).empty)
t30 = fresh(); cc_ = [t30.ingest(bf(flow_id="DUP"))[0], t30.ingest(bf(flow_id="DUP"))[0]]; tst(30, "Duplicate event handling", "201 then 409", cc_, cc_ == [201, 409])
TR = pd.DataFrame(TESTS); print(TR.to_string(index=False)); print(f"\n{(TR.result == 'PASS').sum()}/{len(TR)} tests passed")

banner("STEP 8  Reports + SIEM-style JSON")
generate_incident_report(ids); n, sample = export_siem(ids)
print("Saved reports/incident_report.md and reports/siem_alerts.jsonl"); print(f"{n} alerts exported. Sample SIEM event:\n{json.dumps(sample, indent=2)}")
print("Files:", ["data/network_traffic.csv", "ids_events.db", "ids_dashboard.html"])

banner("STEP 9  SOC dashboard")
_html = render_dashboard(ids); serve_dashboard(8000)        # <- clickable dashboard link appears here
print("\nInline preview (same dashboard):"); display(HTML(_html))

# ---------------- Analyst panel (interactive; needs ipywidgets, preinstalled in Colab) ----------------
def alert_html(aid):
    r = ids.conn.execute("SELECT a.*, f.source_ip, f.destination_ip, f.destination_port, f.protocol FROM alerts a JOIN network_flows f USING(flow_id) WHERE alert_id=?", (aid,)).fetchone()
    if not r: return ""
    notes = "".join(f"<li><code>{n['created_at']}</code> <b>{n['note_type']}</b> - {n['note']}</li>" for n in ids.conn.execute("SELECT * FROM incident_notes WHERE alert_id=? ORDER BY note_id", (aid,))) or "<li>No notes yet</li>"
    esc = lambda s: str(s).replace("&", "&amp;").replace("<", "&lt;")
    return (f"<div style='background:#111A2F;color:#EAF0FA;border:1px solid #24324F;border-radius:12px;padding:14px 18px;font:14px system-ui;max-width:760px'>"
            f"<b style='font-size:16px'>{r['alert_id']}</b> &middot; {esc(r['alert_type'])} &middot; <b>{r['severity']}</b> &middot; risk {r['risk_score']} &middot; status <b>{r['status']}</b><br>"
            f"<span style='color:#A5B4CF'>{r['source_ip']} &rarr; {r['destination_ip']}:{r['destination_port']} {r['protocol']}</span><br><br>{esc(r['description'])}"
            f"<ul style='margin:8px 0 0 0'>{notes}</ul></div>")

def analyst_panel():
    try: import ipywidgets as W
    except ImportError: print("ipywidgets not available - use ids.set_status(alert_id, status, note) and ids.add_note(alert_id, text)."); return
    dd = W.Dropdown(description="Alert", layout=W.Layout(width="760px")); st = W.Dropdown(options=STATUSES, description="Set status")
    note = W.Textarea(description="Note", placeholder="Analyst / resolution note...", layout=W.Layout(width="760px", height="70px"))
    save, refresh = W.Button(description="Save update", button_style="primary"), W.Button(description="Refresh dashboard", button_style="info")
    info, msg, dash = W.Output(), W.Output(), W.Output()
    def opts(keep=None):
        rows = ids.conn.execute("SELECT alert_id,severity,risk_score,alert_type,status FROM alerts ORDER BY (status IN ('NEW','INVESTIGATING')) DESC, risk_score DESC LIMIT 80").fetchall()
        dd.options = [(f"{r['alert_id']} | {r['severity']} | {r['risk_score']:.0f} | {r['alert_type']} | {r['status']}", r["alert_id"]) for r in rows]
        if keep: dd.value = keep
    def show(_=None):
        with info: clear_output(wait=True); display(HTML(alert_html(dd.value)))
    def do_save(_):
        aid = dd.value; cur = ids.conn.execute("SELECT status FROM alerts WHERE alert_id=?", (aid,)).fetchone()["status"]
        code, body = ids.set_status(aid, st.value, note.value or None) if st.value != cur else ids.add_note(aid, note.value)
        with msg: clear_output(wait=True); print(code, body)
        note.value = ""; opts(aid); show()
    def do_refresh(_):
        with dash: clear_output(wait=True); display(HTML(render_dashboard(ids)))
    dd.observe(show, "value"); save.on_click(do_save); refresh.on_click(do_refresh); opts(); show()
    display(HTML("<h3 style='font-family:system-ui;margin:18px 0 6px'>&#128269; Analyst investigation panel</h3>"))
    display(W.VBox([dd, info, st, note, W.HBox([save, refresh]), msg, dash]))
analyst_panel()


======== STEP 1  Synthetic dataset ========
Saved data/network_traffic.csv  (6,000 rows)  |  labels: {'NORMAL': 4800, 'SUSPICIOUS': 1200}


scenario_type,HIGH_CONNECTION_RATE,HIGH_TRAFFIC_VOLUME,MULTI_PORT_PROBING_PATTERN,NORMAL_DATABASE,NORMAL_DNS,NORMAL_EMAIL,NORMAL_SSH,NORMAL_WEB,REPEATED_FAILED_CONNECTIONS,SYN_HEAVY_PATTERN,UNUSUAL_PORT_ACTIVITY
flows,200,200,200,466,971,479,501,2383,200,200,200



======== STEP 2  Feature engineering + ML training (real metrics) ========
Train 4,500 / Test 1,500 (stratified).  Baseline learned from normal training flows only.


,accuracy,precision,recall,f1
model,,,,
Logistic Regression,0.9547,0.9203,0.8467,0.8819
Random Forest,0.9627,0.9388,0.8700,0.9031
Isolation Forest (unsupervised),0.9393,0.8773,0.8100,0.8423


,accuracy,precision,recall,f1
model,,,,
Signature rules only,1.0000,1.0000,1.0000,1.0000
Anomaly only (score >= 50),0.8853,1.0000,0.4267,0.5981
ML only (Random Forest),0.9627,0.9388,0.8700,0.9031
Hybrid (risk >= 41),0.9720,1.0000,0.8600,0.9247


...

,incident_id,source_ip,alert_type,alert_count,first_seen,last_seen,max_severity,max_risk,alert_ids
0,INC-0001,203.0.113.45,Many Destination Ports,12,2026-10-05 19:40:23,2026-10-05 19:40:30,CRITICAL,87.0,"[ALT-10100, ALT-10101, ALT-10102, ALT-10103, A..."
1,INC-0002,192.0.2.131,Many Destination Ports,14,2026-10-05 19:40:12,2026-10-05 19:40:23,CRITICAL,86.9,"[ALT-10077, ALT-10078, ALT-10079, ALT-10080, A..."
2,INC-0003,203.0.113.45,Many Destination Ports,12,2026-10-05 19:23:28,2026-10-05 19:23:39,CRITICAL,86.9,"[ALT-10027, ALT-10028, ALT-10030, ALT-10029, A..."
3,INC-0004,203.0.113.150,Many Destination Ports,10,2026-10-05 19:43:04,2026-10-05 19:43:12,CRITICAL,86.9,"[ALT-10122, ALT-10123, ALT-10124, ALT-10125, A..."
4,INC-0005,203.0.113.201,Many Destination Ports,6,2026-10-05 19:33:56,2026-10-05 19:34:01,CRITICAL,86.9,"[ALT-10055, ALT-10056, ALT-10057, ALT-10058, A..."
5,INC-0006,203.0.113.12,Many Destination Ports,8,2026-10-05 19:22:17,2026-10-05 19:22:23,CRITICAL,86.8,"[ALT-10010, ALT-10011, ALT-10012, ALT-10013, A..."



======== STEP 6  Safe detection scenarios A-G (records only) ========


,conn_rate,fail_ratio,syn,uniq_ports,rules,anomaly,ml_pct,risk,classification,expected
scenario,,,,,,,,,,
A Normal HTTPS,1.5,0.00,3,1,-,0.0,0,0.0,NORMAL,No alert
B Normal DNS,2.0,0.00,0,1,-,0.3,2,0.6,NORMAL,No alert
C Repeated failures,2.9,0.75,60,1,IDS-002,23.6,100,69.1,POTENTIAL INTRUSION,Repeated Failed Connections
D High connection rate,53.2,0.01,217,1,IDS-001,73.6,100,84.1,POTENTIAL INTRUSION,High Connection Rate
E Multi-port pattern,1.0,1.00,1,15,"IDS-003, IDS-005",76.2,100,86.9,POTENTIAL INTRUSION,Many Destination Ports
F SYN-heavy,5.7,0.00,496,1,IDS-004,40.1,100,74.0,POTENTIAL INTRUSION,SYN-Heavy Behaviour
G High volume,0.1,0.00,2,1,IDS-006,66.8,100,74.0,POTENTIAL INTRUSION,Abnormally High Traffic Volume



======== STEP 7  Automated tests (30) ========
 id                                scenario                               expected                                           actual result
T01                         Normal TCP flow                       NORMAL, no rules                                        NORMAL []   PASS
T02                         Normal UDP flow                                 NORMAL                                           NORMAL   PASS
T03                         Normal DNS flow                                 NORMAL                                           NORMAL   PASS
T04                       Normal HTTPS flow                             risk <= 20                                              0.1   PASS
T05                    High connection rate                                IDS-001                                      ['IDS-001']   PASS
T06             Repeated failed connections                                IDS-002                                    


Inline preview (same dashboard):
